# Metrics for recommender systems

_Metrics & Evaluation — measuring models, data & statistics_

**Score a list of suggestions: are the good items near the top, and is the catalog still alive?**

Three families of questions, three families of metrics.

       1. Is the order good? (ranking metrics) — the user sees a list; did you put the items they like near the top? Precision@k, Recall@k, NDCG@k, MAP, MRR, Hit Rate.

---

This notebook is a practice scaffold for the **Metrics for recommender systems** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns** — each sample is an 8x8 grid of pixel intensities (0–16).

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print("image array:", digits.images.shape, " labels:", digits.target.shape)
samples = list(zip(digits.images, digits.target))
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — scikit-learn

### Step 1 — Ranking metrics on one user's ranked listA recommender shows the user an ordered list. We encode the result as `rel`, where `rel[i] = 1` if the item in slot `i` is one the user actually likes. From that single vector we can read several ranking scores.**Precision@k** asks what fraction of the top `k` slots are relevant; **Recall@k** asks what fraction of *all* the user's liked items made it into the top `k`. **Hit Rate@k** is the yes/no version: did at least one relevant item show up? **DCG/NDCG** go further and reward putting relevant items *near the top* by discounting lower positions.

In [ ]:
# rel[i] = 1 if the item at position i is relevant to the user, else 0
rel = np.array([1, 0, 1, 0, 1])   # top-5 list, relevant at slots 1, 3, 5
n_relevant_total = 3              # |R|: how many items the user actually likes

def precision_at_k(rel, k):
    relevant_in_top_k = rel[:k].sum()
    return relevant_in_top_k / k

def recall_at_k(rel, k, n_relevant_total):
    relevant_in_top_k = rel[:k].sum()
    return relevant_in_top_k / n_relevant_total

def hit_rate_at_k(rel, k):
    found_any = rel[:k].sum() > 0
    return 1.0 if found_any else 0.0

def dcg_at_k(rel, k):
    rel = rel[:k]
    positions = np.arange(2, len(rel) + 2)   # log2(i+1), with i starting at 1
    discounts = np.log2(positions)
    return float((rel / discounts).sum())

def ndcg_at_k(rel, k, n_relevant_total):
    ideal = np.ones(min(k, n_relevant_total))   # perfect order: all relevant first
    idcg = dcg_at_k(ideal, k)
    return dcg_at_k(rel, k) / idcg if idcg > 0 else 0.0

k = 5
print("precision@5", round(precision_at_k(rel, k), 3))
print("recall@5   ", round(recall_at_k(rel, k, n_relevant_total), 3))
print("hit_rate@5 ", hit_rate_at_k(rel, k))
print("ndcg@5     ", round(ndcg_at_k(rel, k, n_relevant_total), 3))

### Step 2 — MRR and rating-prediction errors**MRR** (Mean Reciprocal Rank) cares only about *where the first relevant item lands*: if it's at position `r`, the score is `1/r`, then we average over users. It rewards getting one good answer to the very top.When the model predicts a numeric **rating** instead of a ranking, we score it with regression errors: **RMSE** (squares the gaps, so big misses dominate) and **MAE** (averages the absolute gaps, treating every star equally).

In [ ]:
# MRR: reciprocal rank of the FIRST relevant item, averaged over users
def mrr(list_of_rel):
    reciprocal_ranks = []
    for r in list_of_rel:
        hit_positions = np.nonzero(r)[0]
        if len(hit_positions):
            first_hit = hit_positions[0]
            reciprocal_ranks.append(1.0 / (first_hit + 1))
        else:
            reciprocal_ranks.append(0.0)
    return float(np.mean(reciprocal_ranks))

print("MRR        ", round(mrr([rel, np.array([0, 1, 0])]), 3))

# Rating-prediction metrics: how far off are the predicted star ratings?
y_true = np.array([5.0, 3.0, 4.0])
y_pred = np.array([4.0, 1.0, 4.0])

squared_errors = (y_pred - y_true) ** 2
rmse = np.sqrt(np.mean(squared_errors))

absolute_errors = np.abs(y_pred - y_true)
mae = np.mean(absolute_errors)

print("RMSE       ", round(float(rmse), 3), " MAE", round(float(mae), 3))

### Step 3 — Beyond accuracy: coverage and diversityA recommender can score perfectly on ranking yet still be unhealthy if it shows everyone the same few items. **Catalog coverage** measures what fraction of the whole catalog ever gets recommended across all users.**Intra-list diversity** looks *inside one list*: it averages the pairwise dissimilarity (`1 - cosine similarity`) of the items shown, so a list of near-identical items scores low.

In [ ]:
# Catalog coverage: what fraction of the whole catalog ever gets recommended
catalog_size = 1000
recommended_items = {3, 17, 42, 88, 91, 3, 17}   # union over ALL users' lists
unique_recommended = set(recommended_items)
coverage = len(unique_recommended) / catalog_size
print("coverage   ", round(coverage, 3))

# Intra-list diversity = average pairwise dissimilarity (1 - cosine) within one list
item_vecs = np.array([[1, 0, 0], [0, 1, 0], [0.9, 0.1, 0], [0, 0, 1]], dtype=float)
norms = np.linalg.norm(item_vecs, axis=1, keepdims=True) + 1e-9
item_vecs = item_vecs / norms                    # unit-norm rows -> dot product = cosine

sims = item_vecs @ item_vecs.T                    # pairwise cosine similarity matrix
n = len(item_vecs)
upper = np.triu_indices(n, k=1)                   # each unordered pair counted once
diversity = float((1 - sims[upper]).mean())
print("diversity  ", round(diversity, 3))

## Visualize the data & results

_Run a REAL item-item recommender on the digits images: as we show more items (larger k), how do Precision@k, Recall@k, and NDCG@k trade off — and does personalizing beat a popularity baseline on catalog coverage?_

### Step 1 — Build a real item-item recommender on the digits imagesTo get real ranked lists, we treat each query image as a "user" and recommend the most cosine-similar *other* images. We unit-normalise every image first, so a dot product equals cosine similarity, then build one ranked list per query.An item counts as **relevant** when it is the same digit class as the query — a concrete, real-data stand-in for "the user liked it".

In [ ]:
from sklearn.datasets import load_digits

rng = np.random.default_rng(0)
digits = load_digits()
X = digits.data.astype(float)
y = digits.target

# Unit-normalise rows so a dot product equals cosine similarity.
row_norms = np.linalg.norm(X, axis=1, keepdims=True) + 1e-9
X = X / row_norms

# Pick 200 query images to act as 'users'.
idx = rng.choice(len(X), 200, replace=False)

# Similarity of each query to every image, then rank best-first.
S = X[idx] @ X.T                 # 200 x 1797 cosine similarities
for i, qi in enumerate(idx):
    S[i, qi] = -np.inf           # never recommend the query image itself
order = np.argsort(-S, axis=1)   # each row: image indices ranked best-first

### Step 2 — Sweep k and watch precision, recall, and NDCG trade offAs we show more items (larger `k`), **precision** tends to fall (later slots are less reliable) while **recall** rises (we catch more of the relevant items). **NDCG** tracks how well the relevant items are ordered within the top `k`.We compute all three at several values of `k`, averaging over the 200 query users.

In [ ]:
def dcg(rels):
    rels = np.asarray(rels, float)
    positions = np.arange(2, len(rels) + 2)
    return np.sum(rels / np.log2(positions))

ks = [1, 3, 5, 10, 20, 50]
prec, rec, ndcg = [], [], []

for k in ks:
    ps, rs, ns = [], [], []
    for i, qi in enumerate(idx):
        topk = order[i, :k]
        rel = (y[topk] == y[qi]).astype(float)        # relevant = same digit class
        n_rel = (y == y[qi]).sum() - 1                # total relevant, minus the query

        ps.append(rel.sum() / k)
        rs.append(rel.sum() / n_rel)

        idcg = dcg(np.ones(min(k, n_rel)))            # perfect ordering's DCG
        ns.append(dcg(rel) / idcg if idcg > 0 else 0.0)

    prec.append(np.mean(ps))
    rec.append(np.mean(rs))
    ndcg.append(np.mean(ns))

print("k         ", ks)
print("precision ", [round(v, 3) for v in prec])   # [1.0, 0.99, 0.984, 0.967, 0.939, 0.865]
print("recall    ", [round(v, 3) for v in rec])    # [0.006, 0.017, 0.028, 0.054, 0.105, 0.242]
print("ndcg      ", [round(v, 3) for v in ndcg])   # [1.0, 0.992, 0.987, 0.975, 0.953, 0.892]

### Step 3 — Personalized coverage vs a popularity baselineAccuracy isn't the whole story. Here we compare **catalog coverage** of our personalized recommender (each user's top-10) against a **popularity baseline** that shows the same globally-most-similar 10 items to everyone.The personalized recommender touches a large slice of the catalog; the popularity baseline touches almost none — the classic collapse onto a handful of blockbusters.

In [ ]:
N = len(X)

# Personalized: union of every user's top-10 recommendations.
recset = set()
for i in range(len(idx)):
    recset.update(order[i, :10].tolist())
cov_model = len(recset) / N                       # ~0.62

# Popularity baseline: the SAME globally-most-similar 10 items shown to everyone.
avgsim = (X @ X.T).mean(0)                         # average similarity of each item to all
popset = set(np.argsort(-avgsim)[:10].tolist())
cov_pop = len(popset) / N                          # ~0.006

print("coverage personalized", round(cov_model, 3), " popularity", round(cov_pop, 3))

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** A top-3 list for a user whose relevant set is $R=\{P,Q,R,S\}$ ($|R|=4$) comes out as $[Q,\ Z,\ S]$. Compute Precision@3, Recall@3, HR@3, and MRR.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Relevant flags: $Q$ is in $R$ -> 1; $Z$ is not -> 0; $S$ is in $R$ -> 1. So $\text{rel}=[1,0,1]$. — _Mark each shown item against the ground-truth relevant set $R$._
- Precision@3 $=\frac{1+0+1}{3}=\frac{2}{3}\approx 0.67$. — _Fraction of the 3 shown slots that are relevant._
- Recall@3 $=\frac{2}{|R|}=\frac{2}{4}=0.5$. — _Of the 4 items the user likes, 2 made the top 3._
- HR@3 $=1$ and MRR $=\frac{1}{1}=1.0$. — _At least one relevant item appeared (HR), and the first relevant item $Q$ is at position 1, so $1/r=1$._

**Answer:** Precision@3 $\approx 0.67$, Recall@3 $=0.5$, HR@3 $=1$, MRR $=1.0$.

</details>

**Problem 2.** Your model scores NDCG@10 = 0.81 (up from 0.78) but catalog Coverage dropped from 0.40 to 0.06 and Personalization fell sharply. What is happening, and which metric proves it?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Read the accuracy signal: NDCG rose, so the items shown are well-ordered and relevant. — _Accuracy alone looks like an improvement — this is the trap._
- Read the beyond-accuracy signals: Coverage 0.40 -> 0.06 means only 6% of the catalog is ever recommended; falling Personalization means users increasingly see the SAME items. — _These are the symptoms of collapse onto blockbusters / the long-tail problem._
- Conclude: the model is maximizing hits by recommending a tiny set of popular head items to everyone. — _Head items reliably score hits, so a pure-accuracy objective pushes the model there._

**Answer:** The recommender has collapsed onto blockbusters (popularity bias / long-tail starvation). Coverage (0.40 -> 0.06) and the drop in Personalization prove it — the accuracy gain (NDCG) hid the collapse. Fix: set floors on Coverage / Diversity and reward Novelty / Serendipity.

</details>

**Problem 3.** For a star-rating model, true ratings are $[4, 5, 2]$ and predictions are $[4, 4, 5]$. Compute MAE and RMSE, and say why they differ.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Errors: $|4-4|=0$, $|5-4|=1$, $|2-5|=3$. So absolute errors $=[0,1,3]$. — _Both metrics start from the per-item gaps._
- MAE $=\frac{0+1+3}{3}=\frac{4}{3}\approx 1.33$. — _MAE averages the absolute gaps, treating every star of error equally._
- RMSE $=\sqrt{\frac{0^2+1^2+3^2}{3}}=\sqrt{\frac{10}{3}}=\sqrt{3.33}\approx 1.83$. — _Squaring makes the size-3 miss dominate, so RMSE > MAE._

**Answer:** MAE $\approx 1.33$, RMSE $\approx 1.83$. RMSE is larger because squaring punishes the single big (3-star) miss far more than MAE does.

</details>